# Phase 4 – Modelling

**CRISP-DM Step 4 of 5**

**Primary objective**: Identify the firm-level and contextual factors that influence the
*chance* and *intensity* of resource efficiency (RE) action adoption among European SMEs.

RE action adoption is now the **dependent variable**. Firm characteristics, strategic
orientation, perceived barriers, and country/sector context are the **predictors**.

```
Level 3: EU Macro-region  (Northern / Western / Southern / CEE)
  └── Level 2: Country    (27 EU member states)
        └── Level 1: Firm (n ≈ 14,000 EU SMEs)
Cross-classified: Sector  (4 NACE groups)
```

**Analysis roadmap:**

| Section | Goal |
|---------|------|
| 4.3 | Outcome distributions |
| 4.4 | **Bivariate predictor screening** — correlations, Kruskal-Wallis, Cramér's V |
| 4.5 | VIF collinearity check → final predictor set |
| 4.6 M0 | Null models — ICC justification |
| 4.7 M1 | Intensity: `re_action_count ~ predictors + (1\|country)` |
| 4.8 M2 | M1 + macro-region FEs (3-level) |
| 4.9 M3 | Cross-classified: country × sector |
| 4.10 M4 | Random slope — does top predictor effect vary by country? |
| 4.11 M5 | Binary GLMM: `re_any_action ~ predictors + (1\|country)` |
| 4.12 | **Per-action models** — 9 logistic GLMMs + coefficient heatmap |
| 4.13 | Bundle models |
| 4.14 | Sensitivity checks |
| 4.15 | Model comparison table |

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats as scipy_stats
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

from src.data.loader import load_processed
from src.features.engineer import (
    RE_DUMMY_COLS, RE_ACTION_LABELS, RE_ACTION_DUMMY_NAMES,
    BUNDLE_COLS, RE_BUNDLES,
)
from src.models.multilevel import (
    fit_null_model, fit_linear_mixed, fit_linear_random_slope,
    fit_cross_classified, fit_logistic_mixed, fit_ordinal_logit,
    compare_models, likelihood_ratio_test,
    plot_residuals, random_effects_caterpillar, plot_fixed_effects,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

## 4.1 Load Model-Ready Dataset

In [ ]:
df = load_processed('sme_model_ready.parquet')
print(f"Shape: {df.shape[0]:,} rows \u00d7 {df.shape[1]} columns")
df.head(3)

## 4.2 Variable Setup

In [ ]:
from src.features.engineer import (
    RE_DUMMY_COLS, RE_ACTION_LABELS, RE_ACTION_DUMMY_NAMES,
    BUNDLE_COLS, RE_BUNDLES,
    BARRIER_COLS, BARRIER_LABELS,
)

# ── Outcomes ────────────────────────────────────────────────────────────────
OUTCOME_INTENSITY = 're_action_count'
OUTCOME_ANY       = 're_any_action'
ACTION_OUTCOMES   = RE_DUMMY_COLS
BUNDLE_OUTCOMES   = BUNDLE_COLS

# ── Candidate predictors for screening ─────────────────────────────────────
STRUCT_PREDS    = ['firm_size_ord', 'firm_age_ord', 'turnover_bracket_ord']
STRATEGIC_PREDS = ['has_green_products', 'has_climate_strategy', 'uses_renewables']
BARRIER_COUNT_PRED = ['difficulty_count']
INDIVIDUAL_BARRIERS = BARRIER_COLS   # barrier_finance … barrier_tech
MARKET_PREDS    = ['ms_products', 'ms_services']  # dummies; ref = 'both'
PRODUCT_PREDS   = ['recycled_materials_ord', 'has_lifespan_policy']
INVEST_PRED     = ['re_investment_ord']

CANDIDATE_PREDS = (
    STRUCT_PREDS + STRATEGIC_PREDS +
    BARRIER_COUNT_PRED + INDIVIDUAL_BARRIERS +
    MARKET_PREDS + PRODUCT_PREDS + INVEST_PRED
)
CANDIDATE_CAT = ['macro_region', 'sector_label', 'market_scope']

# ── Grouping structure ───────────────────────────────────────────────────────
GROUP_COUNTRY = 'country_label'
GROUP_MACRO   = 'macro_region'
GROUP_SECTOR  = 'sector_label'

# ── Readable labels ─────────────────────────────────────────────────────────
inv_name_map  = {v: k for k, v in RE_ACTION_DUMMY_NAMES.items()}
ACTION_LABELS = {col: RE_ACTION_LABELS[inv_name_map[col]] for col in RE_DUMMY_COLS}

PRED_LABELS = {
    'firm_size_ord':          'Firm size',
    'firm_age_ord':           'Firm age',
    'turnover_bracket_ord':   'Turnover level',
    'has_green_products':     'Green products',
    'has_climate_strategy':   'Climate strategy',
    'uses_renewables':        'Uses renewables',
    'difficulty_count':       'Barrier count (agg)',
    'ms_products':            'Market: products (vs both)',
    'ms_services':            'Market: services (vs both)',
    'recycled_materials_ord': 'Recycled materials*',
    'has_lifespan_policy':    'Lifespan policy*',
    're_investment_ord':      'RE investment (ord)**',
    **BARRIER_LABELS,
}

# ── Analysis sample ──────────────────────────────────────────────────────────
# CORE_PREDS intentionally excludes barrier variables.
# Rationale: barrier questions (q7) are only asked to RE adopters — they are
# structurally missing (MNAR) for non-adopters. Including them in REQUIRED would
# silently drop all non-adopters from df_main, making re_any_action always 1 and
# rendering M5 (binary adoption model) undefined.
#
# Barriers are included in CANDIDATE_PREDS for bivariate screening (with NaN for
# non-adopters handled by nan_policy='omit') and in primary intensity models
# where rows with missing barriers are dropped automatically — effectively
# conditioning the barrier analysis on adoption, which is correct.
CORE_PREDS = STRUCT_PREDS + STRATEGIC_PREDS + MARKET_PREDS

REQUIRED = [OUTCOME_INTENSITY, OUTCOME_ANY] + CORE_PREDS + [GROUP_COUNTRY, GROUP_MACRO]

# Encode market_scope as dummies (ref = 'both') before building df_main
df = df.copy()
df['ms_products'] = (df['market_scope'] == 'products').astype(float)
df['ms_services'] = (df['market_scope'] == 'services').astype(float)

df_main = df.dropna(subset=[c for c in REQUIRED if c in df.columns]).reset_index(drop=True)

adopter_frac = df_main[OUTCOME_ANY].mean()
print(f"Analysis sample (all firms): {len(df_main):,} firms")
print(f"Countries: {df_main[GROUP_COUNTRY].nunique()} | "
      f"Macro-regions: {df_main[GROUP_MACRO].nunique()} | "
      f"Sectors: {df_main[GROUP_SECTOR].nunique()}")
print(f"RE adopters: {adopter_frac*100:.1f}%  |  Non-adopters: {(1-adopter_frac)*100:.1f}%")
print(f"market_scope distribution:\n{df_main['market_scope'].value_counts().to_string()}")
print(f"\nProduct-variable subsample (n2/n3 non-missing): "
      f"{df.dropna(subset=['recycled_materials_ord','has_lifespan_policy']).shape[0]:,}")

## 4.3 Outcome Distributions

In [ ]:
print("RE action count distribution:")
print(df_main[OUTCOME_INTENSITY].value_counts().sort_index().to_string())
print(f"\nMean: {df_main[OUTCOME_INTENSITY].mean():.2f} | "
      f"Median: {df_main[OUTCOME_INTENSITY].median():.0f} | "
      f"SD: {df_main[OUTCOME_INTENSITY].std():.2f}")
print(f"Firms with 0 actions: {(df_main[OUTCOME_INTENSITY]==0).mean()*100:.1f}%")
print(f"Firms with ≥1 action:  {df_main[OUTCOME_ANY].mean()*100:.1f}%")
print(f"\nAdoption rate per action (%):")
print((df_main[ACTION_OUTCOMES].mean() * 100).round(1)
      .rename(ACTION_LABELS).sort_values(ascending=False).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: count histogram
axes[0].hist(df_main[OUTCOME_INTENSITY], bins=range(11), align='left',
             edgecolor='white', color='steelblue')
axes[0].set_xlabel('Number of RE actions adopted (0–9)')
axes[0].set_ylabel('Number of firms')
axes[0].set_title('Intensity: RE Action Count')
axes[0].set_xticks(range(10))

# Right: per-action adoption rates
rates = (df_main[ACTION_OUTCOMES].mean() * 100).rename(ACTION_LABELS).sort_values()
axes[1].barh(rates.index, rates.values, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Adoption rate (%)')
axes[1].set_title('Per-Action Adoption Rates')

plt.tight_layout()
plt.savefig('../reports/figures/adoption_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.4 Bivariate Predictor Screening

For each candidate predictor we compute its association with `re_action_count` (intensity):

- **Spearman ρ** for ordinal/continuous predictors
- **Point-biserial r** (= Pearson) for binary predictors
- **Kruskal-Wallis H** for nominal categorical predictors (sector, macro-region)

We also report association with `re_any_action` (binary any-adoption) for reference.

Predictors are retained if they pass a **Bonferroni-corrected p < 0.05** threshold
(`α / n_tests`). `re_investment_ord` is flagged separately due to directional ambiguity.

In [ ]:
from statsmodels.stats.multitest import multipletests
from src.models.multilevel import apply_fdr

y_count = df_main[OUTCOME_INTENSITY].values
y_any   = df_main[OUTCOME_ANY].values

# Screen core candidates (no structural missing)
core_candidates = [p for p in CANDIDATE_PREDS
                   if p not in PRODUCT_PREDS + INVEST_PRED and p in df_main.columns]
n_tests    = len(core_candidates)
alpha_bonf = 0.05 / n_tests

rows = []
for pred in core_candidates:
    x_raw = df_main[pred]
    x = pd.to_numeric(x_raw, errors='coerce').values  # coerce object cols to float
    if np.isnan(x).all():
        print(f"  Skipping {pred}: all-NaN after coercion (non-numeric column)")
        continue
    rho, p_spear = scipy_stats.spearmanr(x, y_count, nan_policy='omit')
    r_any, p_any = scipy_stats.pointbiserialr(
        y_any[~np.isnan(x)], x[~np.isnan(x)]
    )
    rows.append({
        'predictor':    PRED_LABELS.get(pred, pred),
        'spearman_rho': round(rho, 3),
        'p_count':      round(p_spear, 4),
        'r_any':        round(r_any, 3),
        'p_any':        round(p_any, 4),
    })

screen = pd.DataFrame(rows).sort_values('p_count').reset_index(drop=True)
n_tests = len(screen)
alpha_bonf = 0.05 / n_tests

# Multiple testing correction
# ─────────────────────────────────────────────────────────────────────────────
# Bonferroni (FWER, α/n): appropriate for confirmatory pre-specified tests.
#   Conservative when predictors are positively correlated — all survey items
#   share common causes, so Bonferroni overstates the effective number of
#   independent tests and inflates Type II error.
#
# BH-FDR (q-value): controls expected false discovery rate.
#   Valid under positive dependence (PRDS condition), which holds for our
#   correlated survey predictors (Benjamini & Yekutieli 2001).
#   Recommended threshold for exploratory screening: q < 0.10
#   (not 0.05 — BH at 0.05 is already conservative; 0.10 is standard for
#   discovery-phase research that will be validated in primary models).
#   Reference: Benjamini & Hochberg (1995), Storey & Tibshirani (2003).
_, p_bonf, _, _ = multipletests(screen['p_count'].values, method='bonferroni')
_, p_fdr,  _, _ = multipletests(screen['p_count'].values, method='fdr_bh')

screen['p_bonf']     = p_bonf.round(4)
screen['p_fdr_bh']   = p_fdr.round(4)
screen['sig_bonf']   = screen['p_bonf']   < 0.05   # FWER α = 0.05
screen['sig_fdr']    = screen['p_fdr_bh'] < 0.10   # FDR q = 0.10

print(f"Bonferroni threshold:  p < 0.05/{n_tests} = {alpha_bonf:.4f}")
print(f"BH-FDR threshold:      q < 0.10")
print(f"Predictors significant under Bonferroni:  {screen['sig_bonf'].sum()}/{n_tests}")
print(f"Predictors significant under BH-FDR q<0.10: {screen['sig_fdr'].sum()}/{n_tests}")
print()

# MNAR note: barrier variables are only observed for RE adopters
print("⚠  Barrier variables conditioned on adoption: excluded from M5 (re_any_action).")
print("   Q7 is a hard skip (0 non-adopters answer it). Barriers measure difficulties")
print("   *during* implementation, not barriers *to starting* adoption.")
print()
display(screen)

In [ ]:
# Categorical predictors: Kruskal-Wallis H (count) + Cramér's V (any-adoption)
from scipy.stats import kruskal
import itertools

cat_rows = []
for cat in CANDIDATE_CAT:
    groups = [df_main.loc[df_main[cat] == g, OUTCOME_INTENSITY].dropna().values
              for g in df_main[cat].unique()]
    h_stat, p_kw = kruskal(*groups)
    # Cramér's V: χ² / (n * (min(r,c) - 1))
    ct = pd.crosstab(df_main[cat], df_main[OUTCOME_ANY])
    chi2, p_chi, dof, _ = scipy_stats.chi2_contingency(ct)
    n_ct = ct.values.sum()
    cramers_v = np.sqrt(chi2 / (n_ct * (min(ct.shape) - 1)))
    cat_rows.append({
        'predictor':   cat,
        'KW_H':        round(h_stat, 2),
        'p_KW':        round(p_kw, 4),
        'cramers_V':   round(cramers_v, 3),
        'p_chi2':      round(p_chi, 4),
    })

cat_screen = pd.DataFrame(cat_rows)
print("Categorical predictor screening (vs re_action_count and re_any_action):")
display(cat_screen)

In [ ]:
# Visualise bivariate associations (Spearman ρ with re_action_count)
screen_plot = screen.copy()
screen_plot['label'] = screen_plot['predictor']
screen_plot = screen_plot.sort_values('spearman_rho')

colors = ['steelblue' if p < alpha_bonf else ('orange' if p < 0.05 else 'lightgrey')
          for p in screen_plot['p_count']]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(screen_plot['label'], screen_plot['spearman_rho'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel("Spearman ρ with RE action count")
ax.set_title("Bivariate Predictor–Outcome Association\nBlue = Bonferroni-significant, Orange = p<0.05, Grey = n.s.")

# Add asterisks
for bar, sig in zip(bars, screen_plot['sig_bonf']):
    if sig:
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                sig, va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/predictor_screening.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.5 VIF Collinearity Check → Final Predictor Set

We exclude `re_investment_ord` from the primary models due to directional ambiguity
(treated as sensitivity-only). We exclude any predictor with VIF > 5.

In [ ]:
# VIF on all candidates except the endogenous re_investment_ord
# Restrict to numeric columns only (market_scope dummies are already numeric)
vif_preds = [p for p in CANDIDATE_PREDS
             if p != 're_investment_ord' and p in df_main.columns]
vif_data  = df_main[vif_preds].select_dtypes(include='number').dropna()
vif_preds = list(vif_data.columns)  # update to only numeric

vif_df = pd.DataFrame({
    'predictor': vif_preds,
    'VIF': [variance_inflation_factor(
                sm.add_constant(vif_data.values.astype(float)), i + 1
            ) for i in range(len(vif_preds))],
}).sort_values('VIF', ascending=False)

print("Variance Inflation Factors:")
display(vif_df.round(2))

high_vif = vif_df[vif_df['VIF'] > 5]['predictor'].tolist()
if high_vif:
    print(f"\n⚠ High VIF (>5): {high_vif} — consider dropping from primary models")
else:
    print("\nAll VIF < 5 — no collinearity concern.")

In [ ]:
# Final predictor set:
# Keep all Bonferroni-significant candidates (p < alpha_bonf) excluding re_investment_ord
# If none are eliminated by VIF, retain all significant ones.
sig_preds_raw = screen[screen['p_count'] < alpha_bonf]['predictor'].tolist()
# Map back to column names
label_to_col  = {v: k for k, v in PRED_LABELS.items()}
PREDICTORS = [label_to_col[p] for p in sig_preds_raw
              if label_to_col.get(p) not in (['re_investment_ord'] + high_vif)]

# Safety fallback: if screening is too aggressive, use all non-endogenous candidates
if len(PREDICTORS) < 2:
    PREDICTORS = vif_preds

print("Final PREDICTORS for primary models:")
for p in PREDICTORS:
    print(f"  {p:30s}  ({PRED_LABELS.get(p, p)})")
print(f"\nTotal: {len(PREDICTORS)} predictors")
print("\nNote: re_investment_ord is used only in sensitivity section (Section 4.14).")

### 4.11 M5: Binary adoption model (re_any_action ~ firm characteristics)
#
# Predicts whether a firm has adopted ANY RE action vs. none.
#
# KEY DESIGN DECISION — Barrier variables are EXCLUDED from M5:
# Q7 barrier questions (q7.1–q7.12) are only asked to firms that have already
# undertaken ≥1 RE action (skip pattern: q1.11 ≠ 1). Non-adopters' barriers
# are entirely unobserved. Including barrier variables in a model predicting
# adoption (re_any_action) would introduce structural selection bias — we can
# only compare adopters on their reported barriers, not adopters vs. non-adopters.
# Barriers are valid predictors of *intensity* (how much, given any adoption)
# but NOT of *initiation* (whether adoption occurred at all).

from src.models.multilevel import fit_logistic_mixed

# Adoption predictors: firm structural + strategic + market (no barriers)
adoption_preds = STRUCT_PREDS + STRATEGIC_PREDS + MARKET_PREDS
adoption_preds = [p for p in adoption_preds if p in df_main.columns]

df_m5 = df_main.dropna(subset=[OUTCOME_ANY] + adoption_preds + [GROUP_COUNTRY]).copy()
print(f"M5 sample: {len(df_m5):,} firms | predictors: {adoption_preds}")

m5 = fit_logistic_mixed(
    df_m5,
    outcome=OUTCOME_ANY,
    fixed_effects=adoption_preds,
    group=GROUP_COUNTRY,
    model_name="M5_logistic_adoption",
)
print(m5.summary())

In [ ]:
### 4.12 Per-action models: which firm characteristics predict each RE action type?
#
# Fits 9 logistic GLMMs, one per RE action.
# MULTIPLE TESTING: At α=0.05, fitting 9 models naively yields E[FP] = 0.45.
# We apply BH-FDR correction across p-values for each predictor across models.
#
# Barrier variables EXCLUDED here because they are only observed for adopters
# (firms with ≥1 RE action); using them to predict a specific action type
# would condition on an endogenous intermediary.
#
# Note: fit_logistic_mixed uses BinomialBayesMixedGLM (variational Bayes).
# Frequentist p-values are approximated as: z = fe_mean/fe_sd, p = 2Φ(-|z|).
# This is equivalent to treating the posterior SD as a standard error under a
# normal approximation, which is appropriate for large n (N ≈ 12,000 firms).

from src.models.multilevel import fit_logistic_mixed, apply_fdr
from scipy import stats as _sp_stats

# Predictors valid for ALL firms (no barriers — MNAR)
adopter_safe_preds = [p for p in PREDICTORS if p not in BARRIER_COLS + ['difficulty_count']]

action_results = {}
for action_col in ACTION_OUTCOMES:
    action_name = ACTION_LABELS.get(action_col, action_col)
    m = fit_logistic_mixed(
        df_main,
        outcome=action_col,
        fixed_effects=adopter_safe_preds,
        group=GROUP_COUNTRY,
        model_name=f"logistic_{action_col}",
    )
    action_results[action_name] = m

print(f"Fitted {len(action_results)} per-action logistic GLMMs")
print()

# Extract coefficients and approximate p-values
def extract_logistic_results(m):
    """Extract (coefs, pvalues) from BinomialBayesMixedGLM or standard GLM result."""
    res = m.result
    if hasattr(res, 'fe_mean') and hasattr(res, 'fe_sd'):
        # BinomialBayesMixedGLM: posterior z-score approximation
        names = res.model.fep_names
        b = pd.Series(res.fe_mean, index=names)
        z = res.fe_mean / np.where(res.fe_sd > 0, res.fe_sd, np.nan)
        p = pd.Series(2 * (1 - _sp_stats.norm.cdf(np.abs(z))), index=names)
        return b, p
    elif hasattr(res, 'fe_params') and hasattr(res, 'pvalues'):
        return res.fe_params, res.pvalues
    elif hasattr(res, 'params') and hasattr(res, 'pvalues'):
        return res.params, res.pvalues
    return None, None

pval_rows = {}
coef_rows = {}
for action_name, m in action_results.items():
    b, p = extract_logistic_results(m)
    if b is not None:
        pval_rows[action_name] = p
        coef_rows[action_name] = b

if not pval_rows:
    print("⚠  Could not extract p-values from any per-action model. Check model fitting.")
else:
    pval_df = pd.DataFrame(pval_rows).T    # rows = actions, cols = predictors
    coef_df = pd.DataFrame(coef_rows).T

    # Drop Intercept from display
    pred_cols = [c for c in pval_df.columns if c != 'Intercept']
    pval_df = pval_df[pred_cols]
    coef_df  = coef_df[pred_cols]

    # Apply BH-FDR across all tests (n_actions × n_predictors)
    from statsmodels.stats.multitest import multipletests
    pvals_flat = pval_df.values.flatten()
    # Guard against NaN and empty arrays
    mask = ~np.isnan(pvals_flat)
    padj_flat = np.full_like(pvals_flat, np.nan)
    if mask.sum() > 0:
        _, padj_flat[mask], _, _ = multipletests(pvals_flat[mask], method='fdr_bh')
    padj_df = pd.DataFrame(padj_flat.reshape(pval_df.shape),
                            index=pval_df.index, columns=pval_df.columns)

    print("BH-FDR adjusted p-values (* = q < 0.05, ** = q < 0.01):")
    sig_marker = padj_df.map(
        lambda x: f"{x:.3f}**" if x < 0.01 else (f"{x:.3f}*" if x < 0.05 else f"{x:.3f}")
    )
    display(sig_marker)

In [ ]:
## 4.6 Null Models — ICC Justification

m0_country = fit_null_model(df_main, outcome=OUTCOME_INTENSITY, group=GROUP_COUNTRY)
print()
m0_macro = fit_null_model(df_main, outcome=OUTCOME_INTENSITY, group=GROUP_MACRO)
print()
m0_sector = fit_null_model(
    df_main.dropna(subset=[GROUP_SECTOR]).reset_index(drop=True),
    outcome=OUTCOME_INTENSITY, group=GROUP_SECTOR,
)
print()
print("ICC summary:")
print(f"  Country:      {m0_country.icc:.4f}  ({m0_country.icc*100:.1f}% of variance)")
print(f"  Macro-region: {m0_macro.icc:.4f}  ({m0_macro.icc*100:.1f}% of variance)")
print(f"  Sector:       {m0_sector.icc:.4f}  ({m0_sector.icc*100:.1f}% of variance)")
print()
print("ICC > 0.05 justifies multilevel modelling; ICC > 0.15 indicates country context")
print("is a major driver of RE adoption patterns.")

## 4.7 M1: Intensity Model — Firm Characteristics → RE Action Count (country RI)

`re_action_count ~ PREDICTORS + (1|country)`

Answers: *Which firm-level factors predict how many RE actions a firm adopts,
after accounting for country context?*

In [ ]:
m1 = fit_linear_mixed(
    df_main,
    outcome=OUTCOME_INTENSITY,
    fixed_effects=PREDICTORS,
    group=GROUP_COUNTRY,
    model_name='M1_intensity',
)
print(m1.summary())

## 4.8 M2: M1 + Macro-Region Fixed Effects (3-level)

Add macro-region as a fixed effect so the country random intercepts are centred
within each EU bloc. Shows whether adoption norms differ across blocs beyond
what individual countries explain.

In [ ]:
m2 = fit_linear_mixed(
    df_main,
    outcome=OUTCOME_INTENSITY,
    fixed_effects=PREDICTORS + [f'C({GROUP_MACRO})'],
    group=GROUP_COUNTRY,
    model_name='M2_intensity_macro',
)
print(m2.summary())

## 4.8a M1 Fixed Effects Forest Plot

In [ ]:
def plot_predictor_coefs(model, pred_cols, labels, title='Predictor Effects'):
    res = model.result
    params_avail = [p for p in pred_cols if p in res.fe_params.index]
    coefs = pd.DataFrame({
        'coef':  res.fe_params[params_avail],
        'ci_lo': res.conf_int().loc[params_avail, 0],
        'ci_hi': res.conf_int().loc[params_avail, 1],
        'p':     res.pvalues[params_avail],
    })
    coefs['label'] = [labels.get(p, p) for p in coefs.index]
    coefs = coefs.sort_values('coef')

    fig, ax = plt.subplots(figsize=(8, max(4, len(coefs) * 0.5)))
    colors = ['steelblue' if p < 0.05 else 'lightgrey' for p in coefs['p']]
    ax.barh(coefs['label'], coefs['coef'], color=colors, edgecolor='white')
    ax.errorbar(
        coefs['coef'], coefs['label'],
        xerr=[coefs['coef'] - coefs['ci_lo'], coefs['ci_hi'] - coefs['coef']],
        fmt='none', color='black', capsize=3, linewidth=1,
    )
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Coefficient (effect on RE action count)')
    ax.set_title(f'{title}\nBlue = p<0.05, Grey = n.s.')
    plt.tight_layout()
    return fig

fig = plot_predictor_coefs(m1, PREDICTORS, PRED_LABELS,
                           title='M1: Firm-Level Drivers of RE Adoption Breadth')
plt.savefig('../reports/figures/m1_predictor_effects.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.9 M3: Cross-Classified Model (country × sector)

In [ ]:
df_m3 = df_main.dropna(subset=[GROUP_SECTOR]).reset_index(drop=True)

m3 = fit_cross_classified(
    df_m3,
    outcome=OUTCOME_INTENSITY,
    fixed_effects=PREDICTORS,
    group1=GROUP_COUNTRY,
    group2=GROUP_SECTOR,
    model_name='M3_cross_classified',
)
print(m3.summary())
print(f"\nCombined ICC (country:sector): {m3.icc:.4f}")

## 4.10 M4: Random Slope — Does the Strongest Predictor's Effect Vary by Country?

In [ ]:
avail_preds = [p for p in PREDICTORS if p in m1.result.fe_params.index]
top_predictor = m1.result.fe_params[avail_preds].abs().idxmax()
print(f"Strongest M1 predictor (by |coef|): {top_predictor} ({PRED_LABELS.get(top_predictor, top_predictor)})")
print("Adding a random slope for this predictor across countries...\n")

m4 = fit_linear_random_slope(
    df_main,
    outcome=OUTCOME_INTENSITY,
    fixed_effects=PREDICTORS,
    group=GROUP_COUNTRY,
    random_slope_var=top_predictor,
    model_name='M4_random_slope',
)
print(m4.summary())

## 4.11 M5: Binary GLMM — Drivers of Any RE Adoption (`re_any_action`)

`logit(P(re_any_action=1)) ~ PREDICTORS + (1|country)`

While M1–M4 explain *how many* actions a firm takes, M5 asks: what determines
whether a firm takes *any* action at all (adopter vs. non-adopter)?

In [ ]:
m5 = fit_logistic_mixed(
    df_main,
    outcome=OUTCOME_ANY,
    fixed_effects=PREDICTORS,
    group=GROUP_COUNTRY,
    model_name='M5_any_adoption_glmm',
)
print(m5.summary())

## 4.12 Per-Action Models — What Drives Each Specific RE Action?

For each of the 9 binary RE action dummies we fit a logistic GLMM with country
random intercepts. This reveals which firm characteristics drive *specific* actions
vs. which are universal enablers.

**Key output**: coefficient heatmap — rows = predictors, columns = 9 actions,
colour = log-odds magnitude, stars = statistical significance.

In [ ]:
per_action_models = {}
per_action_coefs  = {}   # pred → action → coef
per_action_pvals  = {}   # pred → action → p-value

for action in ACTION_OUTCOMES:
    print(f"Fitting: {action} ({ACTION_LABELS[action]}) ...", end=' ')
    try:
        m_act = fit_logistic_mixed(
            df_main.dropna(subset=[action]),
            outcome=action,
            fixed_effects=PREDICTORS,
            group=GROUP_COUNTRY,
            model_name=f'M_{action}',
        )
        per_action_models[action] = m_act
        b, p = extract_logistic_results(m_act)
        if b is not None:
            for pred in PREDICTORS:
                per_action_coefs.setdefault(pred, {})[action] = float(b.get(pred, np.nan))
                per_action_pvals.setdefault(pred, {})[action] = float(p.get(pred, np.nan))
        print("OK")
    except Exception as e:
        print(f"FAILED: {e}")

print(f"\nFitted {len(per_action_models)} / {len(ACTION_OUTCOMES)} action models.")

In [ ]:
# Build heatmap matrices
coef_mat = pd.DataFrame(per_action_coefs).T   # rows=predictors, cols=actions
pval_mat = pd.DataFrame(per_action_pvals).T

coef_mat.columns = [ACTION_LABELS[c] for c in coef_mat.columns]
pval_mat.columns = coef_mat.columns
coef_mat.index   = [PRED_LABELS.get(i, i) for i in coef_mat.index]
pval_mat.index   = coef_mat.index

# Significance annotation: *** p<.001, ** p<.01, * p<.05, blank otherwise
annot = pval_mat.map(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
)

fig, ax = plt.subplots(figsize=(12, max(4, len(PREDICTORS) * 0.7)))
sns.heatmap(
    coef_mat.astype(float), annot=annot, fmt='s',
    cmap='RdBu_r', center=0,
    linewidths=0.5, linecolor='white',
    ax=ax, cbar_kws={'label': 'Log-odds coefficient'},
)
ax.set_xlabel('RE Action')
ax.set_ylabel('Firm-Level Predictor')
ax.set_title('Per-Action GLMM Coefficients\n(* p<.05, ** p<.01, *** p<.001)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/per_action_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## 4.13 Bundle Models — Which Factors Drive Each Action Strategy?

In [ ]:
bundle_models = {}
readable_bundle = {
    'bundle_energy':   'Energy transition (energy + renewables)',
    'bundle_circular': 'Circular economy (waste min/sell + recycling + design)',
    'bundle_supply':   'Green supply chain (green suppliers + materials)',
    'bundle_inputs':   'Resource inputs (water + energy + materials)',
}

for bname in BUNDLE_OUTCOMES:
    df_b = df_main.dropna(subset=[bname])
    m_b = fit_linear_mixed(
        df_b, outcome=bname, fixed_effects=PREDICTORS,
        group=GROUP_COUNTRY, model_name=f'M_{bname}',
    )
    bundle_models[bname] = m_b
    print(f"\n{'='*60}")
    print(f"Bundle: {readable_bundle[bname]}")
    print(m_b.summary())

In [ ]:
## 4.14 Sensitivity Checks

### 4.14a With RE investment intensity (endogeneity caveat)

`re_investment_ord` was excluded from primary models due to directional ambiguity.
Here we add it back to M1 to test whether investment mediates the firm-size effect.

In [ ]:
m_invest = fit_linear_mixed(
    df_main.dropna(subset=['re_investment_ord']),
    outcome=OUTCOME_INTENSITY,
    fixed_effects=PREDICTORS + ['re_investment_ord'],
    group=GROUP_COUNTRY,
    model_name='M_sensitivity_investment',
)
print(m_invest.summary())
print("\nNote: positive re_investment_ord coefficient may reflect reverse causation.")

In [ ]:
### 4.14c Product-sector subsample: recycled materials + lifespan policy
# n2 and n3 are only asked to manufacturing/product firms (~63% of sample).
# Run a sensitivity model on this subsample with product variables added.
# We use non-barrier predictors here to avoid restricting the sample to
# adopters-only (barrier vars are MNAR for non-adopters).

non_barrier_preds = [p for p in PREDICTORS if p not in BARRIER_COLS + ['difficulty_count']
                     and p in df.columns]
prod_preds = non_barrier_preds + ['recycled_materials_ord', 'has_lifespan_policy']

df_prod = df.dropna(
    subset=[OUTCOME_INTENSITY] + prod_preds + [GROUP_COUNTRY]
).reset_index(drop=True)

print(f"Product subsample: {len(df_prod):,} firms")
print(f"Adopters: {df_prod[OUTCOME_ANY].mean()*100:.1f}%  |  Predictors: {prod_preds}")

if len(df_prod) < 100:
    print("⚠  Subsample too small for mixed model — skipping.")
else:
    try:
        m_prod = fit_linear_mixed(
            df_prod,
            outcome=OUTCOME_INTENSITY,
            fixed_effects=prod_preds,
            group=GROUP_COUNTRY,
            model_name='M_sensitivity_product_vars',
        )
        print(m_prod.summary())
    except Exception as e:
        print(f"⚠  Model failed: {e}")

print("\nNote: recycled_materials_ord and has_lifespan_policy are structurally missing "
      "for ~37% of firms (services / non-product sectors).")

In [ ]:
### 4.14d Mundlak device — RE consistency check
#
# The Mundlak (1978) device addresses a potential weakness of random-effects
# models: if unobserved country-level factors are correlated with firm-level
# predictors (violating the RE assumption), fixed effects would be preferred.
# The device adds the country-level MEAN of each firm-level predictor as an
# additional fixed covariate.  If the Mundlak terms are jointly non-significant,
# the RE estimator is consistent; if significant, unobserved country-level
# confounders exist and the RE estimator is biased.
#
# Reference: Mundlak (1978), Econometrica 46(1); Schunck (2013) The Stata Journal.
# Python equivalent of R plm::phtest() (Hausman test).

from src.models.multilevel import fit_linear_mixed

# Compute country means of firm-level predictors
mundlak_cols = []
df_mundlak = df_main.copy()
for pred in PREDICTORS:
    if pred in df_mundlak.columns:
        colname = f"{pred}_cmean"
        df_mundlak[colname] = df_mundlak.groupby(GROUP_COUNTRY)[pred].transform("mean")
        mundlak_cols.append(colname)

# Fit M1 augmented with Mundlak terms
mundlak_preds = [p for p in PREDICTORS if p in df_mundlak.columns] + mundlak_cols
df_mundlak = df_mundlak.dropna(subset=[OUTCOME_INTENSITY] + mundlak_preds + [GROUP_COUNTRY])

m_mundlak = fit_linear_mixed(
    df_mundlak,
    outcome=OUTCOME_INTENSITY,
    fixed_effects=mundlak_preds,
    group=GROUP_COUNTRY,
    model_name="M1_Mundlak",
)

# Extract Mundlak term p-values and joint test
from scipy.stats import chi2 as chi2_dist
mundlak_pvals = []
for col in mundlak_cols:
    try:
        p = m_mundlak.result.pvalues.get(col, np.nan)
        if not np.isnan(p):
            mundlak_pvals.append(p)
    except Exception:
        pass

print("Mundlak term p-values (country means of firm predictors):")
for col, pv in zip(mundlak_cols, mundlak_pvals):
    pred_label = PRED_LABELS.get(col.replace('_cmean', ''), col)
    print(f"  {pred_label:30s}: p = {pv:.4f}{'  **' if pv < 0.05 else ''}")

print()
if any(p < 0.05 for p in mundlak_pvals):
    print("⚠  Some Mundlak terms significant → unobserved country-level confounders")
    print("   may bias RE estimates. Interpret fixed effects with caution;")
    print("   consider cross-country FE or sensitivity analysis.")
else:
    print("✓  Mundlak terms jointly non-significant → RE estimator appears consistent.")

In [ ]:
### 4.14e Hurdle count model — two-stage decision process
#
# Theoretical motivation: RE adoption likely involves two separable decisions:
#   Stage 1: Does the firm adopt ANY RE action?  (binary: re_any_action)
#   Stage 2: Given adoption, how many actions?   (truncated count: y | y > 0)
#
# Stage 2 explicitly conditions on adoption, so barrier variables ARE valid
# predictors here — they are only observed for adopters and Stage 2 uses
# only adopters.  We include difficulty_count (aggregate barrier count) as
# the barrier proxy to avoid the multicollinearity of individual barrier dummies.
#
# Reference: Mullahy (1986) J. Econometrics; Cameron & Trivedi (2013) ch. 5.

import statsmodels.api as sm

# ── Stage 1: logistic (all firms — no barriers, MNAR) ─────────────────────
df_s1 = df_main.dropna(subset=[OUTCOME_ANY] + PREDICTORS).copy()

m_hurdle_s1 = sm.formula.logit(
    OUTCOME_ANY + " ~ " + " + ".join(PREDICTORS) + f" + C({GROUP_COUNTRY})",
    data=df_s1,
).fit(disp=False)

print(f"Hurdle Stage 1 (logistic, all firms, n={len(df_s1):,}):")
print(f"  AIC={m_hurdle_s1.aic:.1f} | McFadden R²={1 - m_hurdle_s1.llf/m_hurdle_s1.llnull:.3f}")
print()

# ── Stage 2: Poisson (adopters only, barriers included) ───────────────────
# difficulty_count used as aggregate barrier proxy (avoids individual-barrier VIF).
stage2_preds = PREDICTORS + ['difficulty_count']
df_s2 = df_main[df_main[OUTCOME_ANY] == 1].dropna(
    subset=[OUTCOME_INTENSITY] + stage2_preds
).copy()
df_s2 = df_s2[df_s2[OUTCOME_INTENSITY] > 0]

m_hurdle_s2 = sm.formula.glm(
    OUTCOME_INTENSITY + " ~ " + " + ".join(stage2_preds) + f" + C({GROUP_COUNTRY})",
    data=df_s2,
    family=sm.families.Poisson(),
).fit(disp=False)

print(f"Hurdle Stage 2 (Poisson on adopters, n={len(df_s2):,}):")
print(f"  AIC={m_hurdle_s2.aic:.1f} | deviance/df = {m_hurdle_s2.deviance/m_hurdle_s2.df_resid:.2f}")
print(f"  difficulty_count coef = {m_hurdle_s2.params.get('difficulty_count', float('nan')):.3f}  "
      f"p = {m_hurdle_s2.pvalues.get('difficulty_count', float('nan')):.4f}")
print()

# Coefficient comparison: Stage 2 vs M1 for shared predictors
print("Stage 2 coefficients (log scale) vs M1 linear:")
for p in PREDICTORS:
    if p in m_hurdle_s2.params.index and p in m1.result.fe_params.index:
        b_h = m_hurdle_s2.params[p]
        b_m1 = m1.result.fe_params[p]
        print(f"  {PRED_LABELS.get(p, p):28s}: hurdle-S2={b_h:+.3f}  M1-LMM={b_m1:+.3f}")

In [ ]:
### 4.14f Endogeneity of difficulty_count: direction-of-bias analysis
#        and weighted vs. unweighted coefficient comparison
#
# ── Part A: Direction-of-bias argument for difficulty_count ─────────────────
#
# difficulty_count may be endogenous: firms that have already adopted more RE
# actions may report fewer perceived barriers (they overcame them), creating
# reverse causation.  If so, the OLS estimate of difficulty_count is biased.
#
# Key question: in which DIRECTION?
#
# The expected bias direction:
#   - True effect (causal): barriers → lower adoption (negative)
#   - Reverse causation: more adoption → fewer perceived barriers
#     (attenuation bias — pushes estimated coefficient toward zero)
#   - Omitted variable (firm capability): capable firms adopt more AND face
#     fewer barriers → positive omitted variable bias (attenuation)
# Combined: both reverse causation and OVB attenuate the negative coefficient
# toward zero. If we observe a significant NEGATIVE coefficient on
# difficulty_count, the true barrier effect is at least as large in magnitude
# — the estimate is a lower bound (in absolute value), not an upper bound.
# This is the most favourable situation: endogeneity makes us conservative.
#
# Reference: Solon, Haider & Wooldridge (2015) JHR; VanderWeele & Ding (2017)
#            Annals of Internal Medicine (E-value framework).

# Cross-regression: does re_action_count predict difficulty_count?
# (Suggestive of reverse causation if significant)
import statsmodels.formula.api as smf_

df_endog = df_main.dropna(subset=['difficulty_count', OUTCOME_INTENSITY]).copy()
m_endog_check = smf_.ols(
    f"difficulty_count ~ {OUTCOME_INTENSITY} + firm_size_ord + firm_age_ord + market_scope",
    data=df_endog,
).fit()

b = m_endog_check.params[OUTCOME_INTENSITY]
p = m_endog_check.pvalues[OUTCOME_INTENSITY]
print(f"Cross-regression: re_action_count → difficulty_count")
print(f"  β = {b:.3f}  (p = {p:.4f})")
print()
if p < 0.05 and b < 0:
    print("  ⚠  Negative relationship detected: higher RE adoption → lower barrier count.")
    print("     This is consistent with reverse causation / barrier-overcoming.")
    print("     Expected bias on difficulty_count in M1: attenuation (pushes toward 0).")
    print("     The M1 negative coefficient on difficulty_count is a LOWER BOUND")
    print("     in magnitude — the true effect is at least as large.")
elif p < 0.05 and b > 0:
    print("  ⚠  Positive relationship: higher RE adoption → higher barrier reporting.")
    print("     This could reflect learning-by-doing (adopters discover new barriers).")
    print("     Bias direction is ambiguous; interpret with caution.")
else:
    print("  ✓  No significant reverse causation detected (p ≥ 0.05).")

print()

# ── Part B: Weighted vs. unweighted coefficient comparison ──────────────────
# (Solon, Haider & Wooldridge 2015 diagnostic)
# If w1_sme is a function of covariates in the model (stratified sampling by
# firm size / country / sector), unweighted estimates are unbiased and more
# efficient.  If weights are endogenous (sampling correlated with the outcome),
# WLS is needed to remove bias.
# Decision rule: if weighted and unweighted coefficients agree closely, use
# unweighted for efficiency; if they diverge substantially, weights encode
# relevant DGP information and should be used.

from statsmodels.formula.api import wls, ols

preds_fe = " + ".join(PREDICTORS)
formula_fe = f"{OUTCOME_INTENSITY} ~ {preds_fe} + C({GROUP_COUNTRY})"

df_cmp = df_main.dropna(subset=[OUTCOME_INTENSITY] + PREDICTORS + [GROUP_COUNTRY, 'w1_sme']).copy()

m_ols = ols(formula_fe, data=df_cmp).fit()
m_wls = wls(formula_fe, data=df_cmp, weights=df_cmp['w1_sme']).fit()

# Compare coefficients for predictor variables (exclude country FEs)
cmp_rows = []
for pred in PREDICTORS:
    if pred in m_ols.params and pred in m_wls.params:
        b_ols  = m_ols.params[pred]
        b_wls  = m_wls.params[pred]
        pct_ch = (b_wls - b_ols) / abs(b_ols) * 100 if abs(b_ols) > 1e-8 else np.nan
        cmp_rows.append({
            'predictor':    PRED_LABELS.get(pred, pred),
            'β_unweighted': round(b_ols, 4),
            'β_weighted':   round(b_wls, 4),
            'pct_change':   round(pct_ch, 1) if not np.isnan(pct_ch) else None,
            'sign_flip':    (b_ols > 0) != (b_wls > 0),
        })

cmp_df = pd.DataFrame(cmp_rows).sort_values('pct_change', key=abs, ascending=False)
print("Weighted vs. unweighted coefficient comparison (country FE OLS):")
display(cmp_df)

max_chg = cmp_df['pct_change'].abs().max()
any_flip = cmp_df['sign_flip'].any()
print()
if any_flip:
    print("⚠  Sign flip detected for at least one predictor — weights carry")
    print("   important information; consider survey-weighted models.")
elif max_chg > 20:
    print(f"⚠  Largest change: {max_chg:.0f}% — weights affect some coefficients.")
    print("   Report both weighted and unweighted estimates in appendix.")
else:
    print(f"✓  Max change: {max_chg:.0f}% — weighted and unweighted estimates agree.")
    print("   Unweighted LMM estimates are preferred (more efficient).")

In [ ]:
### 4.14b Poisson GLM sensitivity + overdispersion and zero-inflation checks
#
# re_action_count is a bounded count (0–9). The primary models (M1–M4) treat it
# as quasi-continuous; this section checks whether that approximation is
# defensible by fitting a Poisson GLM and running two diagnostic tests:
#   1. Overdispersion: Var(Y) > E(Y)? If so, negative binomial is preferred.
#   2. Zero-inflation: More zeros than Poisson predicts? If so, hurdle/ZIP model.

import statsmodels.api as sm
from src.models.multilevel import overdispersion_test, zero_inflation_check

# Poisson GLM with country fixed effects (approximates country RE at lower cost)
df_poisson = df_main.dropna(
    subset=[OUTCOME_INTENSITY] + PREDICTORS + [GROUP_COUNTRY]
).copy()
poisson_formula = (OUTCOME_INTENSITY + " ~ " +
                   " + ".join(PREDICTORS) + f" + C({GROUP_COUNTRY})")

m_poisson = sm.formula.glm(
    formula=poisson_formula,
    data=df_poisson,
    family=sm.families.Poisson(),
).fit(disp=False)

print(f"Poisson GLM: n={len(df_poisson):,}, AIC={m_poisson.aic:.1f}, "
      f"deviance={m_poisson.deviance:.1f}, df_resid={int(m_poisson.df_resid)}")

# --- Overdispersion test ---
print("\n--- Overdispersion (Pearson χ²/df) ---")
ypred = m_poisson.predict()
od = overdispersion_test(df_poisson[OUTCOME_INTENSITY].values, ypred)

# --- Zero-inflation check ---
print("\n--- Zero-inflation ---")
zi = zero_inflation_check(df_poisson[OUTCOME_INTENSITY].values, ypred)
print()
print("Interpretation:")
print("  If overdispersed: use negative binomial GLMM (not available in")
print("  statsmodels GLMM; would require R/glmer.nb or Python glmmTMB).")
print("  If zero-inflated: a hurdle or ZIP model is appropriate.")
print("  If neither: the linear mixed model approximation is defensible")

In [ ]:
### 4.14c Ordinal logit on count (treats count as ordered category — no random effects)
m_ord = fit_ordinal_logit(
    df_main.dropna(subset=[OUTCOME_INTENSITY] + PREDICTORS),
    outcome=OUTCOME_INTENSITY,
    predictors=PREDICTORS,
    model_name='sensitivity_ordinal_logit',
)
print(m_ord.summary())

## 4.15 Model Comparison

In [ ]:
primary_models = [m0_country, m0_macro, m0_sector, m1, m2, m3, m4]
comparison = compare_models(primary_models)
display(comparison)

In [ ]:
# LRT: do predictors improve over null?
m0_ml = fit_null_model.__wrapped__(df_main, OUTCOME_INTENSITY, GROUP_COUNTRY) \
    if hasattr(fit_null_model, '__wrapped__') else \
    smf.mixedlm(f'{OUTCOME_INTENSITY} ~ 1', df_main, groups=df_main[GROUP_COUNTRY]).fit(reml=False)

m1_ml_result = smf.mixedlm(
    f'{OUTCOME_INTENSITY} ~ {" + ".join(PREDICTORS)}',
    df_main, groups=df_main[GROUP_COUNTRY],
).fit(reml=False)

from scipy.stats import chi2 as chi2_dist
ll_null = m0_ml.llf if hasattr(m0_ml, 'llf') else m0_country.loglik()
ll_m1   = m1_ml_result.llf
chi2_stat = 2 * (ll_m1 - ll_null)
df_diff   = len(PREDICTORS)
p_lrt     = chi2_dist.sf(chi2_stat, df_diff)
print(f"LRT (null vs M1): chi2({df_diff}) = {chi2_stat:.3f}, p = {p_lrt:.4f}")

## 4.16 Diagnostics — M1 Residuals + Country Random Effects

In [ ]:
fig = plot_residuals(m1)
plt.savefig('../reports/figures/m1_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.17 Country Random Intercepts (Caterpillar Plot)

Which countries have systematically higher/lower RE adoption after controlling for
firm-level characteristics?

In [ ]:
fig = random_effects_caterpillar(m1)
plt.savefig('../reports/figures/country_random_effects.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.18 Results Summary

Fill in after running.

| Model | Outcome | Random structure | AIC | ICC | Top predictor |
|-------|---------|-----------------|-----|-----|---------------|
| M0 null (country) | `re_action_count` | RI: country | | | — |
| M0 null (macro-region) | `re_action_count` | RI: macro | | | — |
| M0 null (sector) | `re_action_count` | RI: sector | | | — |
| M1 intensity | `re_action_count` | RI: country | | | |
| M2 + macro FE | `re_action_count` | RI: country + macro FE | | | |
| M3 cross-classified | `re_action_count` | RI: country×sector | | | |
| M4 random slope | `re_action_count` | RI + RS: country | | | |
| M5 binary GLMM | `re_any_action` | RI: country | — | — | |
| Per-action GLMMs ×9 | each `re_*` dummy | RI: country | — | — | (see heatmap) |
| Bundle models ×4 | each bundle score | RI: country | | | |
| Sensitivity: +investment | `re_action_count` | RI: country | | | |
| Sensitivity: Poisson GLM | `re_action_count` | none | — | — | |